In [ ]:
# Helper utilities for Colab (self-contained)
import math
import numpy as np
from scipy import sparse

def load_sparse_matrix(path: str) -> sparse.csr_matrix:
    matrix = sparse.load_npz(path)
    if not sparse.isspmatrix_csr(matrix):
        matrix = matrix.tocsr()
    return matrix

def _top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 0:
        return np.array([], dtype=np.int64)
    if k >= scores.shape[0]:
        return np.argsort(-scores)
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]

def compute_user_metrics(
    scores: np.ndarray,
    train_items: np.ndarray,
    test_items: np.ndarray,
    k: int,
    use_ndcg: bool = True,
) -> tuple[float, float, float] | None:
    if test_items.size == 0 or k <= 0 or scores.size == 0:
        return None
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    ranked = _top_k_indices(scores, k)
    test_set = set(test_items.tolist())
    hits = sum(1 for item in ranked if item in test_set)
    precision = hits / k
    recall = hits / len(test_set)
    ndcg = 0.0
    if use_ndcg:
        dcg = 0.0
        for rank, item in enumerate(ranked):
            if item in test_set:
                dcg += 1.0 / math.log2(rank + 2)
        ideal = min(len(test_set), k)
        idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_ranking(
    score_fn,
    train_interactions: sparse.csr_matrix,
    test_interactions: sparse.csr_matrix,
    k: int,
    use_ndcg: bool = True,
) -> dict:
    precisions = []
    recalls = []
    ndcgs = []
    num_users = train_interactions.shape[0]
    for user_id in range(num_users):
        test_items = test_interactions[user_id].indices
        if test_items.size == 0:
            continue
        train_items = train_interactions[user_id].indices
        scores = score_fn(user_id)
        metrics = compute_user_metrics(scores, train_items, test_items, k, use_ndcg)
        if metrics is None:
            continue
        precision, recall, ndcg = metrics
        precisions.append(precision)
        recalls.append(recall)
        if use_ndcg:
            ndcgs.append(ndcg)
    result = {
        "k": float(k),
        "users_evaluated": float(len(precisions)),
        "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
    }
    if use_ndcg:
        result["ndcg_at_k"] = float(np.mean(ndcgs)) if ndcgs else 0.0
    return result

def recommend_top_k(scores: np.ndarray, train_items: np.ndarray, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    return _top_k_indices(scores, k)

In [ ]:
# ==========================================
# 1. CÀI ĐẶT THƯ VIỆN & SETUP
# ==========================================
!pip install -q lightfm-next numpy scipy scikit-learn

import os
import numpy as np
import pickle
import time
from scipy import sparse
from lightfm import LightFM

# Cấu hình đường dẫn (Điều chỉnh nếu bạn dùng Google Drive)
DATA_DIR_P5 = "phase5_outputs"
DATA_DIR_P4 = "phase4_outputs"

# ==========================================
# 2. LOAD DỮ LIỆU
# ==========================================
print("--- Đang tải dữ liệu ---")
start_time = time.time()

try:
    train_interactions_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_interactions.npz"))
    train_weights_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_weights.npz"))
    test_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "test_interactions.npz"))

    # Item features dùng CSR để dự đoán nhanh
    item_features = sparse.load_npz(os.path.join(DATA_DIR_P4, "item_features.npz")).tocsr()

    # LightFM training expects COO for interactions/weights.
    train_interactions = train_interactions_csr.tocoo()
    train_weights = train_weights_csr.tocoo()

    print(f"✅ Đã load xong dữ liệu ({time.time() - start_time:.2f}s)")
    print(f"📊 Shape Interactions: {train_interactions_csr.shape}")
    print(f"📊 Shape Item Features: {item_features.shape}")
    print(f"📊 Số lượng tương tác: {train_interactions_csr.nnz}")
except FileNotFoundError as e:
    print(f"❌ Lỗi: Không tìm thấy file. Hãy kiểm tra đường dẫn folder {DATA_DIR_P4} và {DATA_DIR_P5}")
    raise e

# ==========================================
# 3. HUẤN LUYỆN MÔ HÌNH (Aligned với train_hybrid.py)
# ==========================================
print("\n--- Đang khởi tạo và huấn luyện mô hình ---")
np.random.seed(42)

model = LightFM(
    loss="warp",                # Thuật toán tối ưu cho bài toán Ranking
    no_components=64,           # Kích thước vector nhúng (latent factors)
    learning_rate=0.05,
    random_state=42,
 )

# Bắt đầu Fit
train_start = time.time()
model.fit(
    train_interactions,
    sample_weight=train_weights, # Sử dụng trọng số rating thực tế
    item_features=item_features,
    epochs=20,                   # Số vòng lặp huấn luyện (aligned)
    num_threads=4,
 )
print(f"✅ Huấn luyện hoàn tất! Thời gian: {(time.time() - train_start)/60:.2f} phút")

# ==========================================
# 4. ĐÁNH GIÁ (EVALUATION) - aligned với script
# ==========================================
print("\n--- Đang đánh giá hiệu năng (K=10) ---")
eval_start = time.time()

num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def score_fn(user_id: int) -> np.ndarray:
    return model.predict(user_id, item_ids, item_features=item_features)

metrics = evaluate_ranking(
    score_fn,
    train_interactions_csr,
    test_interactions,
    k=10,
    use_ndcg=True,
 )

print("-" * 40)
print(f"🚀 Precision@10: {metrics['precision_at_k']:.6f}")
print(f"🚀 Recall@10:    {metrics['recall_at_k']:.6f}")
print(f"🚀 NDCG@10:      {metrics['ndcg_at_k']:.6f}")
print(f"⏱️ Thời gian đánh giá:   {time.time() - eval_start:.2f}s")
print("-" * 40)

# ==========================================
# 5. LƯU MÔ HÌNH (EXPORT)
# ==========================================
model_filename = "lightfm_model_final.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(model, f)

print(f"\n✅ Đã lưu mô hình thành công vào file: {model_filename}")
print("Bạn có thể tải file này về để sử dụng cho Inference sau này.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.2 MB/s eta 0:00:00
--- Đang tải dữ liệu ---
✅ Đã load xong dữ liệu (0.22s)
📊 Shape Interactions: (79153, 20700)
📊 Shape Item Features: (20700, 1247)
📊 Số lượng tương tác: 1539303

--- Đang khởi tạo và huấn luyện mô hình ---
✅ Huấn luyện hoàn tất! Thời gian: 2.52 phút

--- Đang đánh giá hiệu năng (K=10) ---
----------------------------------------
🚀 Precision@10: 0.021691
🚀 Recall@10:    0.057875
🚀 NDCG@10:      0.044979
⏱️ Thời gian đánh giá:   319.46s
----------------------------------------

✅ Đã lưu mô hình thành công vào file: lightfm_model_final.pkl
Bạn có thể tải file này về để sử dụng cho Inference sau này.


In [ ]:
# Config summary (for reproducibility)
config_summary = {
    "data_dir_p5": DATA_DIR_P5,
    "data_dir_p4": DATA_DIR_P4,
    "loss": "warp",
    "no_components": 64,
    "learning_rate": 0.05,
    "epochs": 20,
    "random_state": 42,
    "num_threads": 4,
    "k_eval": 10,
    "use_ndcg": True,
}

print("--- CONFIG SUMMARY ---")
for key, value in config_summary.items():
    print(f"{key}: {value}")

print("Train shape:", train_interactions_csr.shape)
print("Test shape:", test_interactions.shape)
print("Item features shape:", item_features.shape)

--- CONFIG SUMMARY ---
data_dir_p5: phase5_outputs
data_dir_p4: phase4_outputs
loss: warp
no_components: 64
learning_rate: 0.05
epochs: 20
random_state: 42
num_threads: 4
k_eval: 10
use_ndcg: True
Train shape: (79153, 20700)
Test shape: (79153, 20700)
Item features shape: (20700, 1247)


In [ ]:
import itertools
import pickle
import numpy as np
from lightfm import LightFM

required_vars = ["train_interactions", "train_weights", "train_interactions_csr", "test_interactions", "item_features"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(
        "Missing data variables: " + ", ".join(missing_vars) + ". Run the training/setup cell first."
    )

# ==========================================
# EARLY-STOPPING HYPERPARAM SEARCH
# ==========================================
param_grid = {
    "no_components": [64, 128],
    "learning_rate": [0.01, 0.05],
    "loss": ["warp", "bpr"],
}

MAX_EPOCHS = 50
EVAL_EVERY = 5
PATIENCE = 3
MIN_DELTA = 1e-4
K_EVAL = 10
USE_NDCG = True

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"🚀 Early-stopping search với {len(combinations)} cấu hình...")

best_overall = -1.0
best_params = None
results = []
num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def eval_precision(model):
    def score_fn(user_id: int) -> np.ndarray:
        return model.predict(user_id, item_ids, item_features=item_features)
    metrics = evaluate_ranking(
        score_fn,
        train_interactions_csr,
        test_interactions,
        k=K_EVAL,
        use_ndcg=USE_NDCG,
    )
    return metrics["precision_at_k"], metrics

for i, params in enumerate(combinations):
    print(f"\n[Cấu hình {i+1}/{len(combinations)}] {params}")
    model = LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        random_state=42,
    )
    best_score = -1.0
    best_epoch = 0
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.fit_partial(
            train_interactions,
            sample_weight=train_weights,
            item_features=item_features,
            epochs=1,
            num_threads=4,
        )
        if epoch % EVAL_EVERY != 0:
            continue
        score, metrics = eval_precision(model)
        print(f"  Epoch {epoch}: Precision@{K_EVAL}={score:.6f}")
        if score > best_score + MIN_DELTA:
            best_score = score
            best_epoch = epoch
            best_state = pickle.dumps(model)
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  Early stop (patience={PATIENCE}) at epoch {epoch}")
            break

    if best_state is not None:
        model = pickle.loads(best_state)
    results.append({
        "params": params,
        "best_precision": best_score,
        "best_epoch": best_epoch,
    })

    if best_score > best_overall:
        best_overall = best_score
        best_params = params
        with open("best_lightfm_model.pkl", "wb") as f:
            pickle.dump(model, f)

print("\n" + "="*50)
print("🏆 KẾT QUẢ TỐT NHẤT (Early-Stopping)")
print("="*50)
print(f"✨ Best Precision@{K_EVAL}: {best_overall:.6f}")
print(f"🛠️ Best params: {best_params}")
print("💾 Best model saved to: best_lightfm_model.pkl")

import pandas as pd
df_results = pd.DataFrame([
    {**r["params"], "best_precision": r["best_precision"], "best_epoch": r["best_epoch"]}
    for r in results
 ])
print("\nBảng so sánh chi tiết:")
print(df_results.sort_values(by="best_precision", ascending=False))

🚀 Early-stopping search với 8 cấu hình...

[Cấu hình 1/8] {'no_components': 64, 'learning_rate': 0.01, 'loss': 'warp'}
  Epoch 5: Precision@10=0.010539
  Epoch 10: Precision@10=0.011978
  Epoch 15: Precision@10=0.013020
  Epoch 20: Precision@10=0.013747
  Epoch 25: Precision@10=0.014308
  Epoch 30: Precision@10=0.014896
  Epoch 35: Precision@10=0.015282
  Epoch 40: Precision@10=0.015694
  Epoch 45: Precision@10=0.015943
  Epoch 50: Precision@10=0.016183

[Cấu hình 2/8] {'no_components': 64, 'learning_rate': 0.01, 'loss': 'bpr'}
  Epoch 5: Precision@10=0.001766
  Epoch 10: Precision@10=0.001813
  Epoch 15: Precision@10=0.001942
  Epoch 20: Precision@10=0.002158
  Epoch 25: Precision@10=0.002282
  Epoch 30: Precision@10=0.002393
  Epoch 35: Precision@10=0.002496
  Epoch 40: Precision@10=0.002548
  Epoch 45: Precision@10=0.002605
  Epoch 50: Precision@10=0.002666

[Cấu hình 3/8] {'no_components': 64, 'learning_rate': 0.05, 'loss': 'warp'}
  Epoch 5: Precision@10=0.018458
  Epoch 10: Preci

In [ ]:
import pandas as pd

# 1. Export final model metrics (from cell OvWz2uaMNrjb)
# We convert the 'metrics' dictionary to a DataFrame
if 'metrics' in globals():
    df_metrics = pd.DataFrame([metrics])
    df_metrics.to_csv('final_model_metrics.csv', index=False)
    print("✅ Exported final_model_metrics.csv")

# 2. Export hyperparameter search results (from cell 1Hd68dfqLSpg)
# This saves the comparison table shown at the end of the search
if 'df_results' in globals():
    df_results.sort_values(by='best_precision', ascending=False).to_csv('hyperparameter_search_results.csv', index=False)
    print("✅ Exported hyperparameter_search_results.csv")

✅ Exported final_model_metrics.csv
✅ Exported hyperparameter_search_results.csv
